# LS-NEQ 04_tn_linear_response — Tensornetzwerk und lineare Antwort

**Dossier-Stufe E** (`lsne.experiments.STAGES["E"]`, siehe `docs/DESIGN.md` §8).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Vergleich dreier Antwortebenen auf eine Störung an einer Gitterstelle (`docs/DESIGN.md` §5, D8): (a) gemessene nichtlineare Störung, (b) exakte lineare Antwort per Vorwärts-Autodiff (JVP), (c) faktorisiertes Tensornetzwerk des linearisierten Transformers (eingefrorene Attention-Muster und LN-Skalen, PCA-Unterraum r, Positionsfenster) mit einer von TNCO optimierten Kontraktionsreihenfolge (Fallback opt_einsum). (a)−(b) misst Nichtlinearität, (b)−(c) den Fehler der Faktorisierung. TNCO ist hier ein Optimierer für Kontraktionspfade, kein Interpretierbarkeitsverfahren (siehe `docs/TNCO_INTEGRATION.md`).

**Eingaben.** K=0- und K=8/C1-Varianten der ersten `limits.n_families_tn` Familien, Richtungsbank (lang_dom, ent), `cfg.tn_rank`, `cfg.tn_window`.

**Ausgaben.** Tabellen `tn_compare`, `tn_summary`, `tn_paths`, `tn_basis`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/E/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/04_tn_linear_response/`; am Ende eine kurze `summary_04_tn_linear_response_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → TNCO (optional) → Drive → Konfiguration → Modell → Stufe E → Antwortnormen pro Position → Nichtlinearität/Faktorisierungsfehler → TNCO-vs-opt_einsum-Tabelle → PCA-Basis → Zusammenfassung.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = True   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "04_tn_linear_response"
STAGE = "E"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)
TN_RANK = None                     # PCA-Rang r des Residualraums (Standard cfg.tn_rank = 64; A100: bis d_model)
TN_WINDOW = None                   # Positionsfenster T_w (Standard 32)
from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Stufe-E-Parameter ----------------------------------------------------------------
if TN_RANK is not None:
    cfg.tn_rank = int(TN_RANK)
if TN_WINDOW is not None:
    cfg.tn_window = int(TN_WINDOW)
print(f"Rang r = {cfg.tn_rank}, Fenster T_w = {cfg.tn_window}, Familien für Stufe E: {limits.n_families_tn}, "
      f"TNCO-Schritte: {limits.tn_n_steps} × {limits.tn_n_runs} Läufe, Dosen {limits.tn_doses}")


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def stage_status(stage: str, rd, status: str) -> dict:
    """Report entry of a finished stage: a stale error.txt of an earlier failed run is removed
    (the manifest says ok) and failed substages (G-L, manifest.json -> substages) are listed;
    the status then reads ok_with_substage_errors / reused_with_substage_errors."""
    if C.clear_stale_error(rd.path):
        print(f"Stufe {stage}: veraltete error.txt eines früheren Laufs entfernt ({rd.path})")
    sub = C.substage_status(rd.path, status)
    for name, err in sub["substage_errors"].items():
        print(f"Stufe {stage}: Substufe {name} FEHLER: {err}")
    return sub


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(run_dir=str(rd.path), seconds=0.0, error="", **stage_status(stage, rd, "reused"))
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="",
                             **stage_status(stage, rd, "ok"))
        print(f"Stufe {stage} {report[stage]['status']} in {report[stage]['seconds']:.0f} s: "
              f"{', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Stufe E ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage E --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> [--tn-rank 64 --tn-window 32]
res_E = run_stage("E")
print({k: len(v) for k, v in res_E.items()})


In [ ]:
# --- Antwortnormen pro Fensterposition: gemessen vs. linear (JVP) vs. faktorisiert (TN) --
with guarded("Antwortebenen"):
    tc = table("E", "tn_compare")
    combos = list(dict.fromkeys(zip(tc["tag"], tc["direction"])))[:6]
    epss = sorted(tc["eps"].unique())
    fig, axes = plt.subplots(len(combos), len(epss), figsize=(5.2 * len(epss), 3.1 * len(combos)), squeeze=False, sharex=True)
    for i, (tag, d) in enumerate(combos):
        for j, eps in enumerate(epss):
            g = tc[(tc["tag"] == tag) & (tc["direction"] == d) & (tc["eps"] == eps)].sort_values("rel_pos")
            ax = axes[i, j]
            ax.plot(g["rel_pos"], g["measured_norm"], "o-", ms=3, label="(a) gemessen, nichtlinear")
            ax.plot(g["rel_pos"], g["linear_norm"], "s--", ms=3, label="(b) exakt linear (JVP)")
            ax.plot(g["rel_pos"], g["factorized_norm"], "^:", ms=3, label="(c) faktorisiertes TN")
            ax.set_yscale("log")
            ax.axvline(0, color="k", lw=0.6, ls=":")
            ax.set_title(f"{tag} · {d} · ε={eps:g}", fontsize=9)
            if i == len(combos) - 1:
                ax.set_xlabel("Position relativ zur Injektion")
            if j == 0:
                ax.set_ylabel("‖Antwort‖")
    fig.suptitle("Antwort am Ausgang des letzten Blocks: (a) gemessen, (b) exakt linear, (c) faktorisiertes TN", fontsize=10)
    axes[0, 0].legend(fontsize=7)
    savefig("tn_response_norms")
    plt.show()


In [ ]:
# --- Nichtlinearität |a−b|/|b| und Faktorisierungsfehler |b−c|/|b| -----------------------
with guarded("Nichtlinearität und Faktorisierungsfehler"):
    tc = table("E", "tn_compare")
    ts = table("E", "tn_summary")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    for (tag, d, eps), g in tc.groupby(["tag", "direction", "eps"]):
        g = g[g["rel_pos"] >= 0].sort_values("rel_pos")
        a1.plot(g["rel_pos"], g["nonlinearity"], marker=".", label=f"{tag} {d} ε={eps:g}")
        a2.plot(g["rel_pos"], g["factorization_error"], marker=".", label=f"{tag} {d} ε={eps:g}")
    for ax, t in ((a1, "Nichtlinearität ‖a−b‖/‖b‖ (D8; ∝ ε im linearen Regime)"), (a2, "Faktorisierungsfehler ‖b−c‖/‖b‖")):
        ax.set_yscale("log")
        ax.set_xlabel("Position relativ zur Injektion")
        ax.set_title(t, fontsize=10)
    a1.legend(fontsize=6)
    savefig("tn_errors")
    plt.show()
    if len(ts):
        cols = [c for c in ["tag", "direction", "eps", "r", "T_w", "source_captured", "nonlinearity_wmean", "factorization_error_wmean",
                            "nonlinearity_last", "factorization_error_last", "cos_measured_linear_last", "cos_linear_factorized_last", "path_backend"] if c in ts.columns]
        show(ts[cols])


In [ ]:
# --- Kontraktionspfade: TNCO vs. opt_einsum (Kosten log10 FLOPs, Laufzeit) --------------
with guarded("TNCO vs. opt_einsum"):
    tp = table("E", "tn_paths")
    agg = tp.groupby("backend").agg(backend_used=("backend_used", "first"), n_tensors=("n_tensors", "first"),
                                    T_w=("T_w", "first"), r=("r", "first"), n_blocks=("n_blocks", "first"),
                                    cost_log10_opt_einsum=("cost_log10_opt_einsum", "mean"),
                                    cost_log10_backend=("cost_log10", "mean"), runtime_s=("runtime_s", "mean"),
                                    largest_intermediate_log10=("largest_intermediate_log10", "mean"),
                                    fallback_reason=("fallback_reason", "first"), nbytes=("nbytes", "first"))
    order = [b for b in ["tnco", "opt_einsum", "reference_greedy", "reference_dp"] if b in agg.index]
    agg = agg.reindex(order)
    show(agg)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
    a1.bar(order, agg["cost_log10_opt_einsum"], color=["C3" if b == "tnco" else "C0" for b in order])
    a1.set_ylabel("log10 FLOPs (opt_einsum-Zählung, pro Pfad)")
    a1.set_title("Kontraktionskosten des besten Pfads")
    a2.bar(order, agg["runtime_s"], color=["C3" if b == "tnco" else "C0" for b in order])
    a2.set_ylabel("Optimierer-Laufzeit [s]")
    a2.set_yscale("log")
    a2.set_title("Laufzeit der Pfadsuche")
    for ax in (a1, a2):
        ax.tick_params(axis="x", rotation=20)
    savefig("tn_paths")
    plt.show()
    if "tnco" in agg.index and str(agg.loc["tnco", "fallback_reason"]):
        print("Hinweis: TNCO nicht verwendet ->", agg.loc["tnco", "fallback_reason"])


In [ ]:
# --- PCA-Basis: kumulierte erklärte Varianz des Residualraums an der Injektionsstelle ------
with guarded("PCA-Basis"):
    tb = table("E", "tn_basis")
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.plot(tb["rank"], tb["explained_variance_cum"], "o-", ms=3)
    ax.set_xlabel("Rang r")
    ax.set_ylabel("kumulierte erklärte Varianz")
    ax.set_title(f"Trunkierung auf r={cfg.tn_rank} von d={mb.d_model}")
    ax.axhline(1.0, color="k", lw=0.5, ls=":")
    savefig("tn_basis")
    plt.show()


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    # failed substages (stages G-L) from the manifests: a stage whose table is an empty default
    # must not read plain "ok" in the summary (reused stages included)
    for st, r in report.items():
        if r.get("status") in ("ok", "reused") and r.get("run_dir"):
            r.update(C.substage_status(r["run_dir"], r["status"]))
    substage_errors = {st: r["substage_errors"] for st, r in report.items() if r.get("substage_errors")}
    status_overall = ("error" if any(r.get("status") == "error" for r in report.values())
                      else "substage_errors" if substage_errors else "ok")
    if substage_errors:
        print("ACHTUNG, fehlgeschlagene Substufen:", substage_errors)
    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        status_overall=status_overall, substage_errors=substage_errors,
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
